# Automated GitHub PR Review with Claude

Code review is one of the highest-value activities in software development — and also one of the most time-consuming.
This notebook shows how to use Claude to perform a first-pass review of every pull request automatically.

## Why automate code review?

- **Consistency** — Claude applies the same scrutiny to every PR, regardless of reviewer fatigue.
- **Speed** — feedback arrives in seconds, not hours.
- **Coverage** — Claude can catch classes of issues humans often miss: subtle security flaws, off-by-one errors, missing error handling.

## What Claude can catch

| Category | Examples |
|---|---|
| Bugs | Null-pointer dereferences, logic errors, off-by-one |
| Security | SQL injection, hardcoded secrets, insecure deserialization |
| Performance | N+1 queries, unnecessary allocations, blocking I/O in async code |
| Style | Naming conventions, dead code, overly complex conditionals |
| Architecture | Violation of separation of concerns, missing tests |

This tutorial uses **PyGithub** to fetch PR diffs and post comments, and **Claude** (`claude-opus-4-7`) to do the actual review.

## Install dependencies

In [ ]:
%pip install anthropic PyGithub

## Setup

You need two environment variables:
- `ANTHROPIC_API_KEY` — your Anthropic API key
- `GITHUB_TOKEN` — a GitHub personal access token with `repo` scope


In [ ]:
import os

import anthropic
from github import Github

claude = anthropic.Anthropic(api_key=os.environ["ANTHROPIC_API_KEY"])
gh = Github(os.environ["GITHUB_TOKEN"])

CLAUDE_MODEL = "claude-opus-4-7"
print("Clients initialised.")

## Step 1 — Fetch the PR diff

PyGithub gives us every changed file along with its `patch` — the unified diff for that file.
We collect all patches into a list of `(filename, patch)` tuples.

In [ ]:
def fetch_pr_diff(repo_name: str, pr_number: int) -> list[tuple[str, str]]:
    """
    Fetch changed files for *pr_number* in *repo_name* (e.g. 'owner/repo').

    Returns a list of (filename, patch) tuples.
    Files with no patch (e.g. binary files) are skipped.
    """
    repo = gh.get_repo(repo_name)
    pr = repo.get_pull(pr_number)
    files = pr.get_files()
    diffs = []
    for f in files:
        if f.patch:          # binary files have no patch
            diffs.append((f.filename, f.patch))
    print(f"Fetched {len(diffs)} changed files from PR #{pr_number}")
    return diffs

## Step 2 — Chunk large diffs

A PR that touches many files might exceed Claude's context window.
We split at `diff --git` boundaries so each chunk is a coherent set of file-level diffs, and we keep each chunk under 12,000 characters to leave room for the system prompt and response.

In [ ]:
MAX_CHUNK_CHARS = 12_000

def chunk_diffs(diffs: list[tuple[str, str]]) -> list[str]:
    """
    Combine per-file diffs into chunks that fit inside *MAX_CHUNK_CHARS*.

    Each chunk is a single string containing one or more file diffs.
    If a single file's diff exceeds the limit it is truncated with a notice.
    """
    chunks: list[str] = []
    current = ""

    for filename, patch in diffs:
        header = f"diff --git a/{filename} b/{filename}\n"
        entry = header + patch

        # Truncate single enormous files
        if len(entry) > MAX_CHUNK_CHARS:
            entry = entry[:MAX_CHUNK_CHARS] + "\n... [diff truncated] ..."

        if current and len(current) + len(entry) > MAX_CHUNK_CHARS:
            chunks.append(current)
            current = entry
        else:
            current += entry

    if current:
        chunks.append(current)

    print(f"Split into {len(chunks)} chunk(s)")
    return chunks

## Step 3 — Review each chunk with Claude

We send each diff chunk to Claude with a structured system prompt that tells it exactly what to look for.
Using a numbered list in the system prompt produces consistently structured output that's easy for humans to scan.

In [ ]:
REVIEW_SYSTEM_PROMPT = """
You are a senior software engineer performing a thorough pull-request code review.
For the unified diff provided, produce a structured review with these sections:

1. **Bugs** — logic errors, off-by-one mistakes, null/undefined dereferences.
2. **Security issues** — injection risks, hardcoded secrets, insecure patterns.
3. **Performance concerns** — unnecessary allocations, N+1 queries, blocking I/O.
4. **Style suggestions** — naming, dead code, overly complex conditionals.
5. **Overall assessment** — a short paragraph summarising the quality of the change.

Be specific: reference the filename and line number (from the diff) for each finding.
If a section has no findings, write 'None identified.'.
Do not repeat lines from the diff verbatim; paraphrase or quote briefly.
""".strip()


def review_chunk(diff_chunk: str) -> str:
    """Send one diff chunk to Claude and return its review text."""
    response = claude.messages.create(
        model=CLAUDE_MODEL,
        max_tokens=2048,
        system=REVIEW_SYSTEM_PROMPT,
        messages=[
            {"role": "user", "content": f"Please review the following diff:\n\n```diff\n{diff_chunk}\n```"}
        ],
    )
    return response.content[0].text


def review_all_chunks(chunks: list[str]) -> str:
    """
    Review every diff chunk and concatenate results.
    If there are multiple chunks, each section is prefixed with a chunk header.
    """
    if len(chunks) == 1:
        return review_chunk(chunks[0])

    parts = []
    for i, chunk in enumerate(chunks, 1):
        section = f"### Chunk {i} of {len(chunks)}\n\n{review_chunk(chunk)}"
        parts.append(section)
    return "\n\n---\n\n".join(parts)

## Step 4 — Post the review as a GitHub comment

We format the review in Markdown and post it as an issue comment on the PR.
A collapsible `<details>` block keeps the PR timeline tidy.

In [ ]:
def post_review_comment(repo_name: str, pr_number: int, review_text: str) -> str:
    """
    Post *review_text* as a Markdown comment on the specified PR.
    Returns the URL of the created comment.
    """
    repo = gh.get_repo(repo_name)
    pr = repo.get_pull(pr_number)

    body = (
        "## 🤖 Automated Code Review by Claude\n\n"
        "<details>\n<summary>Click to expand review</summary>\n\n"
        + review_text
        + "\n\n</details>\n\n"
        "---\n*Generated by [claude-review](https://github.com/pintaste/claude-review)*"
    )

    comment = pr.create_issue_comment(body)
    print(f"Review posted: {comment.html_url}")
    return comment.html_url

## Step 5 — Full workflow example

Here we run the complete pipeline against a **hardcoded sample diff** so you can see the output format without making live API calls to GitHub.
Only the Claude API call is real (it uses your `ANTHROPIC_API_KEY`).

In [ ]:
SAMPLE_DIFF = """\
diff --git a/app/auth.py b/app/auth.py
--- a/app/auth.py
+++ b/app/auth.py
@@ -12,8 +12,14 @@ import hashlib

 DB_PASSWORD = "s3cr3t_passw0rd"  # TODO: move to env

-def get_user(username):
-    query = f"SELECT * FROM users WHERE username = '{username}'"
-    return db.execute(query).fetchone()
+def get_user(username, password):
+    query = f"SELECT * FROM users WHERE username = '{username}' AND password = '{password}'"
+    result = db.execute(query).fetchone()
+    if result == None:
+        return False
+    token = hashlib.md5(username.encode()).hexdigest()
+    return {"user": result, "token": token}
"""

# Chunk the sample diff (it's small enough to be a single chunk)
sample_chunks = chunk_diffs([("app/auth.py", SAMPLE_DIFF)])

# Run the review — this calls the Claude API
sample_review = review_all_chunks(sample_chunks)

print(sample_review)

Expected output (your exact wording will vary):

```
1. **Bugs**
- `app/auth.py` line 18: `if result == None` should use `is None` (PEP 8 E711).

2. **Security issues**
- `app/auth.py` line 8: `DB_PASSWORD = "s3cr3t_passw0rd"` is a hardcoded secret. Move it to an environment variable.
- `app/auth.py` lines 14-15: The query is constructed with f-string interpolation, making it vulnerable to SQL injection. Use parameterised queries.
- `app/auth.py` line 20: MD5 is cryptographically broken. Use `secrets.token_hex()` for session tokens.

3. **Performance concerns**
- None identified.

4. **Style suggestions**
- Function name `get_user` no longer reflects its expanded responsibility (authentication); consider renaming to `authenticate_user`.

5. **Overall assessment**
This change introduces critical security vulnerabilities (SQL injection and a hardcoded credential) that must be addressed before merging.
```

To run against a real PR, uncomment the cell below and fill in your repo and PR number.

In [ ]:
# Uncomment and fill in to run against a real PR:
#
# REPO   = "your-org/your-repo"
# PR_NUM = 42
#
# diffs  = fetch_pr_diff(REPO, PR_NUM)
# chunks = chunk_diffs(diffs)
# review = review_all_chunks(chunks)
# post_review_comment(REPO, PR_NUM, review)

## GitHub Actions integration

Add this workflow file to your repository as `.github/workflows/pr-review.yml` to trigger the review automatically on every pull request.

```yaml
name: Automated PR Review

on:
  pull_request:
    types: [opened, synchronize]

jobs:
  review:
    runs-on: ubuntu-latest
    permissions:
      pull-requests: write

    steps:
      - uses: actions/checkout@v4

      - name: Set up Python
        uses: actions/setup-python@v5
        with:
          python-version: '3.11'

      - name: Install dependencies
        run: pip install anthropic PyGithub

      - name: Run PR review
        env:
          ANTHROPIC_API_KEY: ${{ secrets.ANTHROPIC_API_KEY }}
          GITHUB_TOKEN: ${{ secrets.GITHUB_TOKEN }}
          PR_NUMBER: ${{ github.event.pull_request.number }}
          REPO: ${{ github.repository }}
        run: python scripts/review_pr.py
```

Store your `ANTHROPIC_API_KEY` under **Settings → Secrets → Actions** in your repository.
`GITHUB_TOKEN` is provided automatically by GitHub Actions.

## Next steps

- **Inline comments** — use `pr.create_review()` with `comments` to attach feedback directly to the changed lines rather than as a single issue comment.
- **Ignore files** — skip generated files (e.g. `package-lock.json`, `*.min.js`) to focus Claude's attention on the real logic.
- **Custom rules** — extend the system prompt with team-specific conventions (naming patterns, mandatory test coverage, etc.).
- **Re-review on push** — the `synchronize` trigger in the Actions YAML already handles new commits pushed to an open PR.
- **Full production service** — see [claude-review](https://github.com/pintaste/claude-review) for a complete GitHub App built on these same building blocks.